In [19]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os
import numpy as np
from stocktrends import Renko

In [20]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [21]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [ ]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [23]:

def atr(DF,n):
    "function to calculate True Range and Average True Range"
    df = DF.copy()
    df['H-L']=abs(df['high']-df['low'])
    df['H-PC']=abs(df['high']-df['close'].shift(1))
    df['L-PC']=abs(df['low']-df['close'].shift(1))
    df['TR']=df[['H-L','H-PC','L-PC']].max(axis=1,skipna=False)
    df['ATR'] = df['TR'].ewm(com=n,min_periods=n).mean()
    return df['ATR']


In [26]:
def renko_DF(DF):
    "function to convert ohlc data into renko bricks"
    df = DF.copy()
    df.reset_index(inplace=True)
    df2 = Renko(df)
    df2.brick_size = 10
    renko_df = df2.get_ohlc_data()
    return renko_df

In [31]:
ohlc = fetchOHLC("TATAMOTORS","5minute",5)


In [32]:
renko = renko_DF(ohlc)

In [34]:
ohlc

,open,high,low,close,volume
date,,,,,
2025-04-02 09:15:00+05:30,674.95,675.35,666.60,666.75,725343
2025-04-02 09:20:00+05:30,666.80,666.85,662.10,664.70,369508
2025-04-02 09:25:00+05:30,664.75,667.50,663.50,667.15,236082
2025-04-02 09:30:00+05:30,667.10,668.70,666.30,667.45,251271
2025-04-02 09:35:00+05:30,667.45,668.40,665.00,665.65,165103
...,...,...,...,...,...
2025-04-04 15:05:00+05:30,613.45,616.30,613.35,616.25,695845
2025-04-04 15:10:00+05:30,615.85,615.85,614.20,614.40,555631
2025-04-04 15:15:00+05:30,614.35,614.70,611.55,612.60,871954


In [33]:
renko

,date,open,high,low,close,uptrend
0,2025-04-02 09:15:00+05:30,650.0,660.0,650.0,660.0,True
1,2025-04-02 10:45:00+05:30,660.0,670.0,660.0,670.0,True
2,2025-04-04 09:15:00+05:30,660.0,660.0,650.0,650.0,False
3,2025-04-04 09:15:00+05:30,650.0,650.0,640.0,640.0,False
4,2025-04-04 09:20:00+05:30,640.0,640.0,630.0,630.0,False
5,2025-04-04 09:50:00+05:30,630.0,630.0,620.0,620.0,False
